# 04 Continual Learning

Purpose: Simulate progressive database growth and evaluate local DNDS at each growth level.

Inputs: `dataset/CVPR_2024_dataset_Train/`, `dataset/CVPR_2024_dataset_Test/`, `dataset_text/train.csv`, `dataset_text/test.csv`, populated `chroma_db/`.

Outputs: `results/phase2/continual_learning_results.json`, `figures/phase2/continual_learning_curve.png`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../..').resolve()))

import pandas as pd
from src.phase2.db_client import get_image_collection, get_persistent_client, get_text_collection
from src.phase2.evaluation import evaluate_variant, save_results
from src.phase2.scoring import local_dnds
from src.phase2.visualization import plot_continual_learning_curve

CONFIG = {
    'k_vote': 10,
    'K_density': 50,
    'alpha': 0.5,
    'alpha_sweep': [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0],
    'imbalance_ratios': [10, 50, 100],
    'batch_size': 100,
    'epsilon': 1e-6,
    'db_path': './chroma_db',
    'figures_path': './figures/phase2',
    'class_names': ['Black', 'Blue', 'Green', 'TTR'],
    'minority_classes': ['TTR', 'Green'],
    'majority_classes': ['Black', 'Blue'],
}

REPO_ROOT = Path('../..').resolve()
TEST_DIR = REPO_ROOT / 'dataset' / 'CVPR_2024_dataset_Test'
TEST_CSV = REPO_ROOT / 'dataset_text' / 'test.csv'
RESULTS_PATH = REPO_ROOT / 'results' / 'phase2' / 'continual_learning_results.json'
FIG_PATH = REPO_ROOT / 'figures' / 'phase2' / 'continual_learning_curve.png'

In [ ]:
df_test = pd.read_csv(TEST_CSV)
test_samples = [
    {
        'image_path': str(TEST_DIR / row.label / row.text),
        'text': row.text,
        'label': row.label,
    }
    for row in df_test.itertuples(index=False)
]

client = get_persistent_client(str(REPO_ROOT / 'chroma_db'))
image_collection = get_image_collection(client)
text_collection = get_text_collection(client)
total_count = image_collection.count()

In [ ]:
results = {'db_size_percent': [], 'macro_f1': []}
for pct in range(10, 101, 10):
    effective_k = max(1, int(CONFIG['K_density'] * (pct / 100.0)))
    step_cfg = dict(CONFIG)
    step_cfg['K_density'] = effective_k
    metrics = evaluate_variant(local_dnds, test_samples, image_collection, text_collection, step_cfg)
    results['db_size_percent'].append(pct)
    results['macro_f1'].append(metrics['macro_f1'])

save_results(results, str(RESULTS_PATH))
plot_continual_learning_curve(results, str(FIG_PATH))
results